## Cell 0. API keys

Paste your keys between the quotes below and run this cell before anything
else. Leave a line as `""` to use whatever is already exported in the
environment instead.

**Two things worth knowing before you paste.** This notebook is regenerated by
`build_q1_nb.py`, which rewrites every cell from source -- so a key typed here
is lost on the next rebuild. And a key typed here is saved inside the `.ipynb`
file, where it can reach git or a shared copy. For a key you intend to keep,
put it in `fourarm/env/keys.local.env` instead, which this cell reads
automatically and which is gitignored and never regenerated.

In [ ]:
# --- Cell 0. API keys. Run first. -------------------------------------------
import os, pathlib

# PASTE BETWEEN THE QUOTES. Leave "" to fall back to the environment or to
# env/keys.local.env.
KEYS = {
    "OPENAI_API_KEY": "",
    "GEMINI_API_KEY": "",
    "ANTHROPIC_API_KEY": "",
}

# An EMPTY value must never be written into the environment. Assigning ""
# unconditionally would blank a key that is already exported correctly, and
# the failure -- a 401 from a variable that is set but empty -- reads nothing
# like "you left the placeholder alone".
for _name, _value in KEYS.items():
    if _value.strip():
        os.environ[_name] = _value.strip()

# The persistent alternative. Same KEY=value format as env/models.env, one
# per line, # for comments. Read only for names not already set, so anything
# pasted above and anything already exported both win over the file.
_here = pathlib.Path.cwd()
_root = next((c for c in [_here] + list(_here.parents)
              if (c / "out").is_dir() and (c / "experiments").is_dir()), None)
_local = _root / "env" / "keys.local.env" if _root else None
if _local and _local.exists():
    for _line in _local.read_text().splitlines():
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _k, _, _v = _line.partition("=")
        _k, _v = _k.strip(), _v.strip().strip("\'\"")
        if _v and not os.environ.get(_k):
            os.environ[_k] = _v

# Report presence, NEVER the value. Printing a key would write it into the
# notebook's saved output, which is the same leak as pasting it into a cell
# and is easier to do by accident.
#
# The report loops over KEYS, so EVERY key the notebook can use needs a row
# there even when it is only ever supplied by keys.local.env. A name missing
# from KEYS still loads from the file, but silently, and a key that loads
# without being reported is indistinguishable from one that did not load.
for _name in KEYS:
    _set = bool(os.environ.get(_name))
    print("%-18s %s" % (_name, "set" if _set else "NOT SET"))
if _local:
    print("%-18s %s" % ("keys.local.env",
                        "read" if _local.exists() else "absent (optional)"))

# Experiment 2, Q1: Derivation

**When the text omits the capability-relevant quantity, can the model obtain it
from the scene?**

Read at rung **N0** only. The other rungs belong to Q3.

Every cell is independently runnable and idempotent. No cell overwrites a paid
run: the runners resume into their output file and skip trials already
answered. Cells that spend money print the call count and refuse to proceed
until `CONFIRM_SPEND` is set to that exact number.

Run this notebook with the working directory set to `fourarm/`, or anywhere
below it -- cell 1 finds the root itself.

In [ ]:
# --- Cell 1. Setup. No model calls. -----------------------------------------
import collections, csv, datetime, hashlib, json, math, os, pathlib, sys

# Find the package root: the directory holding out/ and experiments/.
here = pathlib.Path.cwd()
ROOT = None
for cand in [here] + list(here.parents):
    if (cand / "out").is_dir() and (cand / "experiments").is_dir():
        ROOT = cand
        break
if ROOT is None:
    raise SystemExit("run this from fourarm/ or below: no out/ + experiments/ found")
for p in (str(ROOT), str(ROOT / "ycb")):
    if p not in sys.path:
        sys.path.insert(0, p)

# RE-IMPORT, never reuse. Python caches modules in sys.modules, so running
# this cell a second time in a live kernel keeps whatever was on disk the
# FIRST time it ran. While the ex2 modules are being edited alongside the
# notebook that is a trap: the kernel holds the old vocabulary, and the
# failure surfaces cells later as a design-check assertion naming a face
# that no longer exists, which reads like a code error and is not one.
#
# Dropping the entries and importing fresh is used rather than
# importlib.reload because these modules import each other, and reload
# leaves a half-updated graph unless the order is exactly right.
for _stale in [m for m in list(sys.modules)
               if m.startswith(("experiments.ex2", "analysis.ex2"))
               or m in ("ycb_objects",)]:
    del sys.modules[_stale]

# WHICH KERNEL THIS IS, checked before the first project import.
#
# The very next line reaches core.decision.state_builder through
# mancheck -> vlm_allocator, and that imports numpy; visibility.py, in cell
# 3, needs PIL and scipy. On a kernel without them the notebook dies forty
# lines deep inside somebody else's module with "No module named 'numpy'",
# which reads as a broken repository rather than as a kernel picked from a
# list of six. Checked here, where the answer is one sentence.
_missing = []
for _m in ("numpy", "PIL", "scipy"):
    try:
        __import__(_m)
    except ImportError:
        _missing.append(_m)
if _missing:
    _venv = ROOT.parent / ".venv" / "bin" / "python"
    raise SystemExit(
        "WRONG KERNEL.\n"
        "  This kernel is  %s\n"
        "  and it has no %s.\n"
        "  Use instead     %s\n"
        "  In VS Code: Select Kernel, then Python Environments, then the\n"
        "  interpreter at that path. It is the only one in this tree with\n"
        "  ipykernel AND numpy, PIL and scipy. Several unrelated kernels are\n"
        "  registered on this machine and any of them will get this far and\n"
        "  then fail."
        % (sys.executable, ", ".join(_missing), _venv))

from core.cell import cell_config as C
from core.decision import model_registry as MR
from experiments.ex2 import grade as G
from experiments.ex2 import labels as L
from experiments.ex2 import mancheck as MC
from experiments.ex2 import prompts as P
from experiments.ex2 import run as R
from experiments.ex2 import solo as S
from experiments.ex2 import transforms as T
from experiments.ex2 import visibility as VIS
from analysis.ex2.ex2_stats import newcombe, paired_mean_ci, spans_zero, wilson
# The notebook machinery: loaders, the share definition, the paired
# contrast and the spend gate. In a module rather than in this cell so
# that Q2 and Q3 use the same ones rather than a second copy, and so
# that harness/h_ex2_q_common.py can pin them. What stays in the cells
# is what is a DECISION: the models, the rung, the conditions, the
# usable rule, each cost, and every CONFIRM_SPEND.
from analysis.ex2.ex2_q_common import (Outputs, answered,       # noqa
                                       coupling, fmt, full_flip_count,
                                       is_franka, keep_analysable,
                                       load_run, paired_delta,
                                       paired_diffs, pct,
                                       provenance_row, run_meta,
                                       sha256, share_at, share_counts,
                                       show, spend_gate)

# --- paths ------------------------------------------------------------------
CAPTURES = ROOT / "out" / "ex2_capture_block"
RUNS     = ROOT / "runs"
TABLES   = ROOT / "tables" / "ex2_frameprobe"
FIGURES  = ROOT / "figures" / "ex2_frameprobe"
for d in (RUNS, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

# --- constants, every one read from a source of truth ------------------------
RUNG        = "N0"                       # Q1 is read here and nowhere else
PREFERENCE  = "franka"
CONDITIONS  = ("congruent", "congruent_face")
REPEATS     = 1
FACES       = P.RESTING_FACES            # small_face, large_face
LABEL       = "ycb_block"

FRANKA_MAX  = C.ARM_TYPES["franka"]["max_grasp_m"]
UR_MAX      = C.ARM_TYPES["ur10"]["max_grasp_m"]
DIMS        = T.DIMS_M[LABEL]
FACTS       = T.POSE_FACTS_BY_LABEL[LABEL]

# The registry has no hardcoded model list: aliases() reads FOURARM_MODELS.
try:
    ALIASES = MR.aliases()
except Exception as exc:
    ALIASES = []
    print("model registry unavailable (%s); set MODELS by hand below" % exc)
# THREE models since 2026-08-27. claude-sonnet-5 was added because the
# design needs a third model that CLEARS the two-way face probe: with two
# models, a single failure at cell 5b leaves one, and one model cannot show
# that a result is a property of models rather than of this one model.
# It is not here for being the most capable available; see env/models.env.
#
# claude_md RATHER THAN claude. Same model, claude-sonnet-5, at effort
# medium instead of the API default of high. At the default it read the
# two-way face probe at 65 percent against gpt's 95 and gemini's 100, and
# it failed by BIAS rather than blindness: large_face on 75 percent of
# trials, 90 percent right when the block lies flat and 40 percent when it
# stands. Deliberation is how a prior like "blocks lie flat" gains weight,
# so lower effort is the move that fits the failure. Cell 5b is what tests
# it. The default-effort runs stay on disk under the alias "claude".
#
# gpt_hi RATHER THAN gpt. Same model, gpt-5.6-terra, at reasoning_effort
# high instead of low. Claude runs at the Anthropic default effort of high
# and Gemini Flash exposes no effort control at all, so gpt at low made the
# one model with the LEAST test-time compute the yardstick for the other
# two. Effort is still not matched across providers and cannot be -- that
# stays in Limitations -- but the reasoning models are now on the same
# nominal tier.
#
# The low-effort runs are NOT deleted. runs/ex2_q1_cue2way_gpt_r*.jsonl
# record gpt at reasoning_effort low over this same sample and stay on disk
# as the evidence for what effort was worth here: 95 percent at low. A
# separate alias rather than an edit to GPT_PARAMS is what makes those rows
# still readable, which is the reason env/models.env gives for gpt_hi
# existing at all.
#
# Named rather than taken wholesale from ALIASES. FOURARM_MODELS also lists
# qwen and gpt_hi, and a run's model set must be a decision recorded here,
# not whatever the registry happens to carry. The fallback keeps the same
# three so a registry failure cannot silently shrink the design.
_WANT = ("gpt_hi", "gemini", "claude_md")
MODELS = tuple(a for a in ALIASES if a in _WANT) or _WANT
if set(MODELS) != set(_WANT):
    print("WARNING: %s requested, %s available from the registry. Every "
          "table below is per model, so a missing one narrows the design "
          "rather than breaking it -- but say so in the chapter."
          % (list(_WANT), list(MODELS)))

# --- credentials: reported, not assumed -------------------------------------
# Until 2026-08-27 a hand-added launcher cell started JupyterLab in a browser
# and refused to launch when a key was missing. That cell is gone: it spawned
# a NEW server every time it ran, which is how eight of them accumulated, each
# serving its own in-memory copy of this notebook, so an edit on disk could be
# invisible in the tab you were typing in. VS Code runs the kernel directly
# and needs no launcher -- but the key check it performed was worth keeping,
# so it lives here.
#
# This REPORTS rather than raises. Cells 1-5 and every analysis cell make no
# model calls and must stay runnable with no key at all. What it buys is
# learning about a missing key now instead of at cell 6, part-way into a run.
#
# The usual cause is launching VS Code from Finder or the Dock, which does not
# inherit a login shell, so a key exported in .zshrc is absent here while
# present in any terminal. The message below says so, because the symptom
# otherwise looks like a broken registry.
#
# key_var is read from the registry, never hardcoded: models.env lets each
# alias name its own variable, and a hardcoded OPENAI_API_KEY would check the
# wrong one the moment that is used.
MISSING_KEYS = []
for _alias in MODELS:
    try:
        _var = MR.describe(_alias)["key_var"]
    except Exception as _exc:
        MISSING_KEYS.append("%s: %s" % (_alias, _exc))
        continue
    if not os.environ.get(_var):
        MISSING_KEYS.append("%s: %s is not set" % (_alias, _var))

# Output paths travel together in one object, so a notebook cannot end up
# with a root and a tables directory that disagree. Rebound to bare names
# because every call site below reads better as write_csv(...) than as
# OUT.write_csv(...), and because leaving those call sites untouched is
# what made this extraction verifiable against the tables already on disk.
OUT = Outputs(ROOT, TABLES, FIGURES)
rel, write_csv = OUT.rel, OUT.write_csv

print("root        ", ROOT)
print("captures    ", CAPTURES.relative_to(ROOT), "(exists:", CAPTURES.is_dir(), ")")
print("rung        ", RUNG, " preference", PREFERENCE, " repeats", REPEATS)
print("models      ", MODELS, " (registry knows: %s)" % (ALIASES or "nothing"))
print("prompt ver  ", P.EX2_PROMPT_VERSION)
# Printed, not assumed. If a stale kernel ever slips past the re-import
# above, this is the line that shows it, at the top of the run rather than
# in an assertion twenty cells later.
print("faces       ", FACES, " chance %.1f%%" % (100.0 / len(FACES)))
if MISSING_KEYS:
    print("api keys     MISSING -- analysis runs, model calls will not:")
    for _m in MISSING_KEYS:
        print("               ", _m)
    print("             launch VS Code from a shell that exports them:")
    print("               open -a 'Visual Studio Code' <repo>")
else:
    print("api keys     present for %s" % (", ".join(MODELS),))
print()
print("block           %.3f x %.3f x %.3f m"
      % (DIMS["height"], DIMS["width"], DIMS["depth"]))
print("franka opens to  %.3f m   ur opens to %.3f m" % (FRANKA_MAX, UR_MAX))
print("resting faces   %s" % (FACES,))
for f in FACES:
    print("   %-11s needs %.3f m" % (f, FACTS[f]["grasp_m"]))

## Cell 2. Design check

No model calls. Derives the opening for each resting face from the authored
cuboid dimensions and asserts it matches what `transforms` declares. The prompt
states the bounding-box convention, so a divergence here would make the prompt
wrong rather than silent.

In [ ]:
# --- Cell 2. Design check. No model calls. ----------------------------------
from ycb_objects import YCB as _SPECS      # the authored object dictionary

# STALE-IMPORT GUARD. Cell 1 purges sys.modules before importing, so a
# module edited on disk is picked up whenever cell 1 is re-run. This
# catches the case where cell 1 was NOT re-run -- editing a module and
# jumping straight back to this cell -- and the worse case where the
# NOTEBOOK ITSELF is stale, because JupyterLab holds its own copy in the
# browser and does not re-read the file when it changes underneath. A
# stale cell 1 has no purge, so the modules stay old and the design
# assertion below fails naming a face that no longer exists. That reads
# like a code error and is not one, which is why this checks first and
# says which of the two it is.
#
# The comparison is against the SOURCE ON DISK, not against a constant
# written here, so it stays true across future vocabulary changes.
import re                                   # local: a stale Cell 1 may
                                            # not have imported it
_src = pathlib.Path(P.__file__).read_text()
_on_disk = re.search(r'EX2_PROMPT_VERSION\s*=\s*["\'](.+?)["\']', _src)
if _on_disk and _on_disk.group(1) != P.EX2_PROMPT_VERSION:
    raise SystemExit(
        "STALE IMPORT: this kernel holds prompts.py version %s, but the file "
        "on disk is %s.\n"
        "  Loaded faces: %s\n"
        "  Fix: re-run Cell 1, which drops the cached modules and imports "
        "fresh.\n"
        "  If re-running Cell 1 does not clear it, the NOTEBOOK is stale, not "
        "the kernel:\n"
        "  JupyterLab is running the copy it loaded into the browser. Use "
        "File > Reload\n"
        "  Notebook from Disk, then Restart Kernel and Run All."
        % (P.EX2_PROMPT_VERSION, _on_disk.group(1), list(FACES)))

# Each block prim is spawned already resting on a face, with size PRE-ORIENTED
# to that pose: size is (x, y, z) with z vertical. So the two horizontal
# extents are size[0] and size[1], and the opening is the smaller of them.
# YCB is keyed without the "ycb_" scene prefix.
PRIM_OF_FACE = {L.TRUE_POSE[p]: p for p, lab in L.POSE_ENTRIES.items()
                if lab == LABEL}

design_rows = []
problems = []
for face in FACES:
    prim = PRIM_OF_FACE[face]
    size = _SPECS[prim.replace("ycb_", "")]["size"]
    horiz = sorted(size[:2], reverse=True)          # a = larger, b = smaller
    vertical = size[2]
    opening = min(horiz)
    declared = FACTS[face]["grasp_m"]

    if abs(opening - declared) > 1e-9:
        problems.append("%s: bounding box gives %.3f, transforms declares %.3f"
                        % (face, opening, declared))
    # A real permutation check, all three extents. It compared only the
    # SMALLEST until 2026-08-27, so a prim sized 0.200 x 0.200 x 0.050 --
    # not the block at all -- passed a check whose message said it was
    # verifying a permutation. That matters more with two faces than it
    # did with three: there are fewer cross-checks left, and this cell is
    # what stands between a mis-authored prim and the whole experiment.
    if (sorted(round(v, 6) for v in list(horiz) + [vertical])
            != sorted(round(DIMS[k], 6) for k in ("height", "width", "depth"))):
        problems.append(
            "%s: extents %s are not a permutation of the block %s"
            % (face, sorted(list(horiz) + [vertical]),
               sorted(DIMS[k] for k in ("height", "width", "depth"))))

    franka_ok = declared <= FRANKA_MAX
    ur_ok = declared <= UR_MAX
    design_rows.append([face, "%.3f" % vertical, "%.3f" % horiz[0],
                        "%.3f" % horiz[1], "%.3f" % declared,
                        "%.3f" % FRANKA_MAX, "%.3f" % UR_MAX,
                        franka_ok, ur_ok,
                        "franka, preference satisfied" if franka_ok
                        else "UR, preference overridden"])

# The design only works if the Franka is feasible on one face and not the
# other, and the UR on both. Anything else and Q1 has no contrast.
feasible = [r[0] for r in design_rows if r[7]]
if sorted(feasible) != ["small_face"]:
    problems.append("franka feasible on %s, expected small_face alone"
                    % sorted(feasible))
if not all(r[8] for r in design_rows):
    problems.append("a UR is infeasible somewhere; it must be legal everywhere")

show(["face", "vert", "horiz_a", "horiz_b", "opening", "franka", "ur", "picks"],
     [[r[0], r[1], r[2], r[3], r[4], r[7], r[8], r[9]] for r in design_rows])
print()
if problems:
    raise AssertionError("DESIGN CHECK FAILED:\n  " + "\n  ".join(problems))
print("PASS  every opening is the smaller horizontal extent, and the Franka")
print("      is feasible on small_face and not on large_face.")
print("      A third face, the middle one, was withdrawn on 2026-08-27: it")
print("      was flat like large_face and differed only in geometry, which")
print("      made it the sharper test, but no model read it (GPT 58%,")
print("      Fisher p=0.76 over 81 trials). The cost is that a model")
print("      reading posture and applying a rule can no longer be told")
print("      apart from one deriving the opening from geometry.")

write_csv("tab_ex2_q1_design.csv",
          ["resting_face", "vertical_m", "horiz_a_m", "horiz_b_m",
           "opening_needed_m", "franka_max_m", "ur_max_m", "franka_feasible",
           "ur_feasible", "deriving_model_picks"],
          design_rows)

## Cell 3. Capture inventory and legality

No model calls. Loads the captures, checks the grid is complete, re-asserts the
recorded settle heights, and runs the **real validator** over every scene to
establish which positions can carry the contrast at all.

This cell is the reason the sample is 29 positions rather than 30, and it fails
loudly rather than letting the analysis assume otherwise.

**Why the settle heights are re-checked here.** The prompt never says which flat
orientation to expect. The convention sentence -- *an object resting flat lies on
its largest face* -- was deliberately left out, because capture enforces it
instead: `capture_ex2_scene.py` fails a capture that settles at the wrong height
rather than relabelling it with the face it was asked for. That assertion is real
and it does raise, but it post-dates most of the captures on disk, and the trail
check below it compares only the recorded face *word* against the prim -- never
the height that word is supposed to describe. So the single guarantee standing
behind the prompt's silence was being taken on trust at the point where the data
is actually read. Every capture records `ex2.settled[name]`, so checking it costs
nothing. The tolerance is read out of the capture script's source rather than
typed here, so it cannot drift from the value the captures were accepted under.

In [ ]:
# --- Cell 3. Capture inventory and legality. No model calls. ----------------
import re                                   # local, as in Cell 2: Cell 1
                                            # does not import it, so this
                                            # cell must not depend on Cell 2
                                            # having been run first
scenes = R.load_scenes(str(CAPTURES))          # normalises the idle UR
raw    = R.load_scenes(str(CAPTURES), present_ur=False)   # as written
trail  = [json.loads(l) for l in open(CAPTURES / "consults.jsonl") if l.strip()]

by_pos = collections.defaultdict(dict)
for s in scenes:
    pos, member = s["seq"].rsplit("_", 1)
    prim = [o["name"] for o in s["state"]["objects"] if LABEL.split("_")[-1] in o["name"]][0]
    by_pos[pos][L.TRUE_POSE[prim]] = s

# DERIVED, never literal. This read "90 captures / 30 positions" until
# 2026-08-27 and raised the moment four positions were added to the capture
# plan. A count typed here goes stale silently; one derived from the
# directory cannot. What actually matters is not the total but that every
# position carries every face, which `missing` below checks.
inv_problems = []
if len(scenes) != len(by_pos) * len(FACES):
    inv_problems.append("expected %d captures (%d positions x %d faces), "
                        "found %d" % (len(by_pos) * len(FACES), len(by_pos),
                                      len(FACES), len(scenes)))
missing = {p: sorted(set(FACES) - set(v)) for p, v in by_pos.items()
           if set(v) != set(FACES)}
if missing:
    inv_problems.append("positions missing a face: %s" % missing)
if len({s["seq"] for s in scenes}) != len(scenes):
    inv_problems.append("duplicate seq ids")

# The face is derived from the PRIM, never from the trail's word, and the
# trail is then checked against it.
#
# Captures written before 2026-08-27 record ex2.resting_face in a superseded
# vocabulary where "upright" meant small_face and "small_face" meant the
# retired middle face. capture_ex2_scene.py now writes the geometric name
# directly, so new captures need no translation; this map reads the old ones
# and is why the check is against the prim rather than the word.
TRAIL_FACE = {"upright": "small_face", "small_face": "edge",
              "large_face": "large_face"}

# READ FROM THE CAPTURE SCRIPT, not typed here. capture_ex2_scene.py
# imports isaaclab at module scope and cannot be imported into this kernel,
# and a literal copied into the notebook would go stale the moment the
# tolerance is retuned -- which it was, from 0.010 to 0.005, on 2026-08-27.
# Same regex-the-source trick cell 2 uses for EX2_PROMPT_VERSION.
_cap_src = (ROOT / "ycb" / "capture_ex2_scene.py").read_text()
_tol = re.search(r"^SETTLE_TOL_M\s*=\s*([0-9.]+)", _cap_src, re.M)
if not _tol:
    raise SystemExit(
        "SETTLE_TOL_M not found in ycb/capture_ex2_scene.py. It is the "
        "tolerance the captures were accepted under and the notebook must "
        "not invent one; if it was renamed, update this cell.")
SETTLE_TOL_M = float(_tol.group(1))
for rec in trail:
    prim = [o["name"] for o in rec["state"]["objects"] if "block" in o["name"]][0]
    want = L.TRUE_POSE.get(prim)
    if want is None:
        continue          # a retired-face capture; load_scenes drops it too
    word = (rec.get("ex2") or {}).get("resting_face")
    # BOTH vocabularies are accepted, and only because each is checked
    # against the prim. A word is fine if it already IS the derived face
    # (written 2026-08-27 or later) or if it translates to it (written
    # before). Anything else is a genuine disagreement. Accepting both is
    # not laxity: the prim is the truth in either case, and the word is
    # never the thing consulted downstream.
    if word != want and TRAIL_FACE.get(word) != want:
        inv_problems.append("%s: trail says %r, prim says %r"
                            % (rec["seq"], word, want))

    # THE SETTLE HEIGHTS, RE-ASSERTED WHERE THE DATA IS READ.
    #
    # The prompt says nothing about which flat orientation to expect. The
    # convention sentence ("an object resting flat lies on its largest
    # face") was deliberately NOT added, on the grounds that capture
    # enforces it instead -- and it does: capture_ex2_scene.py raises on a
    # capture that settles at the wrong height rather than labelling it
    # with the face it was asked for. But that assertion post-dates most
    # captures on disk, and the check above compares only the trail's face
    # WORD against the prim, never the height that word is supposed to
    # describe. So the one guarantee standing behind the prompt's silence
    # was, at this point, taken on trust. It is not expensive to check.
    for name, d in (rec.get("ex2") or {}).get("settled", {}).items():
        z, want_z = d.get("z_above_table"), d.get("expected_rest_z")
        if want_z is None:
            inv_problems.append("%s: %s has no expected_rest_z, so its "
                                "resting face was never verified"
                                % (rec["seq"], name))
        elif abs(z - want_z) > SETTLE_TOL_M:
            inv_problems.append(
                "%s: %s settled at z=%.4f, expected %.4f within %.3f. It is "
                "not on the face this capture claims."
                % (rec["seq"], name, z, want_z, SETTLE_TOL_M))

print("captures %d   positions %d   faces per position %s"
      % (len(scenes), len(by_pos),
         sorted({len(v) for v in by_pos.values()})))
print("idle UR presented: %s"
      % dict(collections.Counter(s["idle_ur"] for s in scenes)))
print("idle UR as captured: %s"
      % dict(collections.Counter(
          tuple(sorted(a["name"] for a in s["state"]["arms"]
                       if a["state"] == "IDLE" and a["name"].startswith("ur")))
          for s in raw)))
print()

# --- the real validator, per scene ------------------------------------------
legal = {}
for pos in sorted(by_pos):
    for face, s in by_pos[pos].items():
        st, meta = T.transform({"state": s["state"],
                                "positions_exact": s["positions_exact"]},
                               "congruent")
        tid = R.flip_task_id(s["state"], meta["flip_prim"])
        legal[(pos, face)] = sorted(R.legal_arms(s, meta["flip_prim"], tid))

def has_franka(arms):
    return any(a.startswith("franka") for a in arms)

# TWO independent preconditions, not one. Legality asks whether the aperture
# contrast EXISTS at a position; visibility asks whether the block can be
# SEEN there. A position can carry the full contrast with the block hidden
# behind the Franka, and until 2026-08-27 nothing noticed: e10 presents 3%
# of the median block area and GPT inverted both its posture trials.
#
# visibility.verdict reads pixels only and never a model reply, so a
# position is never excluded for having scored badly.
#
# Run PER PREFIX, not over the directory at once. Each pose is scored
# against the median of its own pose, and the west and east banks sit at
# different distances from the camera: a w block renders about 10 percent
# larger than an e block in the same pose. One pooled median would raise
# the bar for the far bank and lower it for the near one, which is a
# comparison between banks rather than a test of occlusion.
OCCLUDED, _vis_detail = [], {}
for _pfx in sorted({p[0] for p in by_pos}):
    _c = VIS.measure(str(CAPTURES), prefix=_pfx)
    _ok, _bad, _d = VIS.verdict(_c)
    print("visibility, %s bank:" % _pfx)
    print(VIS.report(_d, _ok, _bad))
    print()
    OCCLUDED += _bad
    _vis_detail.update(_d)

USABLE, excluded = [], {}
for pos in sorted(by_pos):
    sm, lg = (legal[(pos, f)] for f in ("small_face", "large_face"))
    ok = has_franka(sm) and lg and not has_franka(lg)
    if ok:
        USABLE.append(pos)
    else:
        excluded[pos] = {"small_face": sm, "large_face": lg}

show(["position", "small_face", "large_face", "usable"],
     [[pos, ",".join(legal[(pos, "small_face")]) or "NONE",
       ",".join(legal[(pos, "large_face")]) or "NONE",
       "yes" if pos in USABLE else "NO"] for pos in sorted(by_pos)])
print()
USABLE = [p for p in USABLE if p not in OCCLUDED]
print("positions carrying the full contrast and showing the block: %d of %d"
      % (len(USABLE), len(by_pos)))
for pos in sorted(set(OCCLUDED)):
    print("  EXCLUDED %s  the block is occluded here (%.2f of the pose"
          % (pos, _vis_detail[pos]["worst"]))
    print("           median, worst in %s). The contrast may exist, but a"
          % _vis_detail[pos]["worst_pose"])
    print("           perception result from a picture that does not show")
    print("           the object is not a result about the model.")
for pos, v in excluded.items():
    print("  EXCLUDED %s  %s" % (pos, v))
    print("           the franka is never legal here, so there is no arm choice")
    print("           to make and no contrast to measure. Excluded with cause,")
    print("           not dropped silently.")

write_csv("tab_ex2_q1_inventory.csv",
          ["position", "usable", "occluded", "idle_ur", "legal_small_face",
           "legal_large_face"],
          [[pos, pos in USABLE, pos in OCCLUDED,
            by_pos[pos]["small_face"]["idle_ur"],
            ";".join(legal[(pos, "small_face")]),
            ";".join(legal[(pos, "large_face")])] for pos in sorted(by_pos)])

if inv_problems:
    raise AssertionError("INVENTORY FAILED:\n  " + "\n  ".join(inv_problems))
if len(USABLE) < 20:
    raise AssertionError("only %d usable positions; the contrast is not "
                         "estimable and the run should not be paid for"
                         % len(USABLE))

# --- what the PAID cells ask about ------------------------------------------
# Defined here, ONCE, and used by both cell 6 and cell 7. Putting the choice
# in each paid cell would let the two be set differently, so congruent and
# dims would cover different scene sets and the paired contrast in cell 10
# would silently compare two different samples.
#
# TRUE is the standing decision: ask about every captured scene, including
# the excluded positions, and drop them in cell 8 at ANALYSIS time. It costs
# a little more and buys something the write-up needs -- the excluded rows
# are in the data, so the exclusion can be shown to predate any accuracy
# result rather than being read as a position dropped for scoring badly.
#
# Set FALSE to pay only for the usable positions. The analysis is unaffected
# either way: cell 8 restricts to USABLE regardless.
RUN_ALL_POSITIONS = True

CALL_SCENES = ([s for s in scenes
                if s["seq"].rsplit("_", 1)[0] in USABLE]
               if not RUN_ALL_POSITIONS else scenes)

print()
print("PASS  inventory complete, %d positions usable." % len(USABLE))
print("      paid cells will ask about %d scenes (%s)"
      % (len(CALL_SCENES),
         "all captured, excluded positions included on purpose"
         if RUN_ALL_POSITIONS else "usable positions only"))

## The probe: does the `named` gloss leak where a face IS stated?

**These two cells make model calls.** Two conditions, three models, one repeat.
Each cell prints its own cost line; set `CONFIRM_SPEND` to that number to let it
run. Roughly 20 minutes per cell.

`congruent` and `congruent_face` both state `resting_face` in the state text.
The `named` gloss additionally says the extents were "measured standing on its
smallest face", which describes `small_face`. If the models are reading that
phrase as a pose claim even when a pose is given, the supplied-face conditions
are partly text-following too, and the reference lines they anchor need
qualifying. If nothing moves, the confound is confined to `dims` and the
correction to the chapter is one subsection.

**Interrupt them freely.** `solo.run` appends and flushes each row as it lands,
and skips every trial already answered when it starts. Re-running a cell resumes
it; re-running a finished cell makes no calls at all and says so.

The `named` twins to compare against are already on disk at three repeats:
`ex2_q1_congruent_N0.jsonl` and `ex2_q1_congruent_face_N0.jsonl`. Nothing here
writes to those files.

In [ ]:
# --- Congruent at N0, EXTENTS frame. MAKES MODEL CALLS. ---------------------
CONG_EXT = RUNS / "ex2_q1_congruent_N0_extents.jsonl"
CONG_NAMED = RUNS / "ex2_q1_congruent_N0.jsonl"

# CALL_SCENES, not scenes: cell 3 decides which positions are paid for, so
# this cell and the next cannot drift apart, and neither can drift from the
# named runs they are compared against.
n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
print("COST: %d scenes x %d models x %d repeat = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
print("      condition congruent, rung %s, dims frame EXTENTS" % RUNG)
print("      named twin already on disk: %s (%d answered)"
      % (CONG_NAMED.name, answered(CONG_NAMED)))

CONFIRM_SPEND = None            # <-- set to the number in the COST line

# factors= is the guard against a later cell rebinding REPEATS: the gate
# refuses when the counts stop multiplying to the number being confirmed,
# rather than the run quietly coming out a third of the size.
if spend_gate(n_calls, CONFIRM_SPEND, CONG_EXT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS))):
    # dims_frames is the ONE thing that differs from the run that produced
    # CONG_NAMED. Everything else -- scenes, models, preference, rung,
    # modality, kind -- is read from cell 1 and cell 3, so the two runs
    # cannot come apart on anything the comparison is not about.
    S.run(str(CAPTURES), out_path=str(CONG_EXT), models=MODELS,
          conditions=("congruent",), preferences=(PREFERENCE,),
          rungs=(RUNG,), modalities=("V",), kind="pair", repeats=REPEATS,
          dims_frames=("extents",))
    print("answered now:", answered(CONG_EXT))

In [ ]:
# --- Congruent-face at N0, EXTENTS frame. MAKES MODEL CALLS. ----------------
FACE_EXT = RUNS / "ex2_q1_congruent_face_N0_extents.jsonl"
FACE_NAMED = RUNS / "ex2_q1_congruent_face_N0.jsonl"

n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
print("COST: %d scenes x %d models x %d repeat = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
print("      condition congruent_face, rung %s, dims frame EXTENTS" % RUNG)
print("      named twin already on disk: %s (%d answered)"
      % (FACE_NAMED.name, answered(FACE_NAMED)))

# THE GLOSS THIS CONDITION GETS UNDER EXTENTS, printed rather than trusted.
# prompts._object_fields dispatched on `condition == "dims"` until this
# notebook was written, which handed congruent_face the FULL gloss -- the one
# that announces an "opening_needed_m" the condition withholds. That would
# have changed two things at once and made this cell unreadable. It is fixed
# in prompts.py; this is the assertion that it stays fixed.
_gloss = P._object_fields("congruent_face", "extents")
assert "opening_needed_m" not in _gloss and "No opening is given" in _gloss, (
    "the extents gloss for congruent_face names an opening the state does "
    "not carry:\n" + _gloss)
assert "smallest face" not in _gloss, (
    "the extents gloss still contains the phrase this probe exists to "
    "remove:\n" + _gloss)
print("      gloss checked: no opening named, no 'smallest face'")

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, FACE_EXT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS))):
    S.run(str(CAPTURES), out_path=str(FACE_EXT), models=MODELS,
          conditions=("congruent_face",), preferences=(PREFERENCE,),
          rungs=(RUNG,), modalities=("V",), kind="pair", repeats=REPEATS,
          dims_frames=("extents",))
    print("answered now:", answered(FACE_EXT))

## Read-out

No model calls. Reads the four files off disk -- two `named`, two `extents` --
so it survives a kernel restart and can be run part-way through a paid cell to
see what has landed so far.

Same helpers Q1 uses (`load_run`, `keep_analysable`, `paired_diffs`,
`paired_mean_ci`), so the `named` columns here should reproduce the numbers
already in the Q1 tables rather than being a second, differently-computed
version of them.

**What to read.** Both conditions state the true face, so opening accuracy is
already high under both frames and will not separate the two explanations. What
separates them is whether anything MOVES when the only thing that changed is a
phrase the condition has made redundant.

The repeat counts differ on purpose: `named` is the existing three-repeat run,
`extents` is one repeat. That is fine for a presence test and is why the `n`
column is printed.

In [ ]:
# --- Read-out. No model calls. ----------------------------------------------
FILES = {
    ("congruent",      "named"):   RUNS / "ex2_q1_congruent_N0.jsonl",
    ("congruent",      "extents"): RUNS / "ex2_q1_congruent_N0_extents.jsonl",
    ("congruent_face", "named"):   RUNS / "ex2_q1_congruent_face_N0.jsonl",
    ("congruent_face", "extents"): RUNS / "ex2_q1_congruent_face_N0_extents.jsonl",
}

# A file that is missing or part-written is REPORTED, not silently rendered as
# a blank row. A probe read part-way through its own paid cell is a normal
# thing to do here, and "extents has not been bought yet" and "extents was
# bought and moved nothing" must not look the same.
loaded, status = {}, []
for (cond, frame), path in sorted(FILES.items()):
    if not path.exists():
        status.append([cond, frame, path.name, 0,
                       len(USABLE) * len(FACES) * len(MODELS),
                       "MISSING", "-"])
        loaded[(cond, frame)] = []
        continue
    rows, skipped = load_run(path, cond, MODELS)
    # trial_id carries the frame, so a file can only hold the frame it was
    # written for -- but check rather than assume: an out_path typo would
    # otherwise pool extents rows into the named column.
    frames = sorted({r.get("dims_frame", "named") for r in rows})
    if frames not in ([frame], []):
        raise AssertionError("%s holds dims_frame %s, expected %r. The two "
                             "frames must not share a file."
                             % (path.name, frames, frame))
    keep = keep_analysable(rows, USABLE)
    reps = sorted(r for r in {x.get("repeat") for x in keep} if r is not None)
    # One repeat's worth is every usable position, both faces, every model.
    # Reported as a RATIO rather than as "complete", because the two frames
    # legitimately hold different repeat counts and a flat complete/partial
    # flag would call the one-repeat extents run short against the
    # three-repeat named one.
    per_rep = len(USABLE) * len(FACES) * len(MODELS)
    status.append([cond, frame, path.name, len(keep), per_rep,
                   "%.2f" % (len(keep) / float(per_rep)),
                   ",".join("r%s" % r for r in reps) or "-"])
    loaded[(cond, frame)] = keep
    if skipped:
        print("  %s: %d rows from models outside MODELS, not counted"
              % (path.name, sum(skipped.values())))

show(["condition", "frame", "file", "analysable", "per repeat",
      "repeats worth", "repeats present"], status)
print()
print("analysable = no transport error, reply parsed, and the position is one")
print("of the %d that carry the contrast and show the block. A run still in" % len(USABLE))
print("progress shows a fractional repeats-worth; that is expected and the")
print("cells below are still readable, just noisier.")
print()

# --- the per-cell numbers ---------------------------------------------------
TOL = 0.006                       # as Q1 cell 12
SMALL_OPEN = FACTS["small_face"]["grasp_m"]

def opening_stats(rows):
    """(n stated, % correct for the face actually shown, % reporting 0.050).

    The third column is the ANCHOR. Under dims it is what the frame moved:
    the gloss names small_face, and a model reading it as a pose claim reports
    that face's opening whatever the picture shows. Correctness and the anchor
    come apart only when the true face is large_face, which is exactly half
    these rows.
    """
    stated = [r for r in rows if r.get("opening_needed_m") is not None]
    if not stated:
        return 0, None, None
    ok = sum(1 for r in stated
             if abs(r["opening_needed_m"] - FACTS[r["face"]]["grasp_m"]) <= TOL)
    anchored = sum(1 for r in stated
                   if abs(r["opening_needed_m"] - SMALL_OPEN) <= TOL)
    return len(stated), pct(ok, len(stated)), pct(anchored, len(stated))

rows_out = []
deltas = {}
for cond in CONDITIONS:
    for model in MODELS:
        for frame in ("named", "extents"):
            sub = [r for r in loaded[(cond, frame)] if r["model"] == model]
            n_open, ok, anchor = opening_stats(sub)
            # THE SAME quantity Q1 reports: Franka share on small_face minus
            # Franka share on large_face, paired within position, positions
            # averaged. Not a pooled difference -- see paired_diffs.
            pairs = paired_diffs(sub, USABLE, "small_face", "large_face")
            diffs = [d for _, d in pairs]
            mean, lo, hi, npos = paired_mean_ci(diffs)
            fk, fn = full_flip_count(diffs)
            deltas[(cond, model, frame)] = mean
            rows_out.append([cond, model, frame, len(sub), n_open,
                             fmt(ok), fmt(anchor),
                             fmt(share_at([r for r in sub
                                           if r["face"] == "small_face"])),
                             fmt(share_at([r for r in sub
                                           if r["face"] == "large_face"])),
                             npos, fmt(mean), fmt(lo), fmt(hi),
                             "%d/%d" % (fk, fn)])

show(["condition", "model", "frame", "rows", "stated", "opening ok%",
      "reports .050%", "franka% small", "franka% large", "pos",
      "delta", "lo", "hi", "full flips"], rows_out)
print()
print("delta = franka share on small_face minus franka share on large_face,")
print("        paired within position over %d positions. 100 is the ceiling:" % len(USABLE))
print("        the Franka every time the block is on its small face and never")
print("        when it is on its large face. 0 is no contrast at all.")
print()

# --- the one number the probe exists to produce -----------------------------
print("=" * 72)
print("FRAME EFFECT: delta(extents) - delta(named), per condition and model")
print("=" * 72)
move_rows = []
for cond in CONDITIONS:
    for model in MODELS:
        dn, de = deltas[(cond, model, "named")], deltas[(cond, model, "extents")]
        gap = None if (dn is None or de is None
                       or dn != dn or de != de) else de - dn
        move_rows.append([cond, model, fmt(dn), fmt(de), fmt(gap)])
show(["condition", "model", "delta named", "delta extents", "extents - named"],
     move_rows)
print()
print("NO INTERVAL ON THE LAST COLUMN, on purpose. It is a difference of")
print("paired differences across two runs of different sizes -- three repeats")
print("against one -- and a t interval on it would be reporting a precision")
print("the design does not have. This is a presence test: under dims the same")
print("quantity moved by roughly 45 points on Gemini at N0 and 30 to 48 on")
print("GPT at the treated rungs. Something of that order is the effect being")
print("looked for; a handful of points is not resolvable here and must not be")
print("written up as one.")
print()
print("=" * 72)
print("WHAT THE TWO OUTCOMES MEAN, stated before the numbers are read")
print("=" * 72)
print("  NOTHING MOVES     the gloss phrase is inert once a face is stated.")
print("                    The confound is confined to dims. Q1's reference")
print("                    lines and all of Q2 stand, and the correction to")
print("                    the chapter is one subsection about dims.")
print()
print("  SOMETHING MOVES   the phrase is read as a pose claim even when a")
print("                    pose is given. The supplied-face conditions are")
print("                    partly text-following, and every reference line")
print("                    drawn from them needs qualifying.")